In [1]:
# 1) Check GPU
!nvidia-smi

Thu May 21 08:43:48 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX PRO 6000 Blac...    Off |   00000000:05:00.0 Off |                    0 |
| N/A   31C    P0             46W /  600W |       0MiB /  97887MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [2]:
# 2) Install dependencies
!pip -q install -U uv

# Basic Python dependencies.
!uv pip install --system -U openai tqdm requests jsonschema psutil

# Install recent vLLM nightly for CUDA 13.0 / Blackwell.
# If this fails in your environment, use the auto backend line below instead.
!uv pip install --system -U vllm --torch-backend=cu130 --extra-index-url https://wheels.vllm.ai/nightly/cu130

# Fallback only if the cu130 line fails:
# !uv pip install --system -U vllm --torch-backend=auto --extra-index-url https://wheels.vllm.ai/nightly

# Version check
import torch
import vllm
import sys

print("Python:", sys.version)
print("Torch:", torch.__version__)
print("Torch CUDA:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)
print("vLLM:", vllm.__version__)

Using Python 3.12.13 environment at: /usr
Resolved 25 packages in 73ms
Checked 25 packages in 0.27ms
Using Python 3.12.13 environment at: /usr
Resolved 187 packages in 7.13s
Checked 187 packages in 1ms
Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
Torch: 2.11.0+cu130
Torch CUDA: 13.0
CUDA available: True
GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition
vLLM: 0.21.1rc1.dev167+gd97ba29fd


In [3]:
# 3) Mount Google Drive and prepare paths
from google.colab import drive
from pathlib import Path
import shutil
import json
import os

drive.mount("/content/drive")

GDRIVE_PROJECT_DIR = Path("/content/drive/MyDrive/final_project/idea_1")
GDRIVE_INPUT_PATH = GDRIVE_PROJECT_DIR / "hotpotqa_docs_chunks.json"

LOCAL_WORK_DIR = Path("/content/hotpotqa_kg_test")
LOCAL_WORK_DIR.mkdir(parents=True, exist_ok=True)

LOCAL_INPUT_PATH = LOCAL_WORK_DIR / "hotpotqa_docs_chunks.json"

KG_DIR = GDRIVE_PROJECT_DIR / "kg"
KG_DIR.mkdir(parents=True, exist_ok=True)

assert GDRIVE_INPUT_PATH.exists(), f"Input file not found: {GDRIVE_INPUT_PATH}"

# Copy to local disk for faster reading.
shutil.copy2(GDRIVE_INPUT_PATH, LOCAL_INPUT_PATH)

print("Input:", LOCAL_INPUT_PATH)
print("Output folder:", KG_DIR)

Mounted at /content/drive
Input: /content/hotpotqa_kg_test/hotpotqa_docs_chunks.json
Output folder: /content/drive/MyDrive/final_project/idea_1/kg


In [4]:
# 4) Start vLLM server - optimized throughput version
import subprocess
import time
import requests
from pathlib import Path
import os
import shlex
import psutil

MODEL_NAME = "Qwen/Qwen3.5-27B"
PORT = 8000
BASE_URL = f"http://localhost:{PORT}/v1"

# Total context length = prompt tokens + output tokens.
# 12288 is enough for ~1k-2k prompt + adaptive output up to 8192.
MAX_MODEL_LEN = 12288

# Keep memory margin on RTX PRO 6000 Blackwell 96GB.
GPU_MEMORY_UTILIZATION = 0.88

# Throughput settings.
MAX_NUM_SEQS = 8
MAX_NUM_BATCHED_TOKENS = 16384

SERVER_LOG_PATH = Path("/content/vllm_server.log")
SERVER_PID_PATH = Path("/content/vllm_server.pid")

def kill_process_tree(pid):
    # Kill a process and its children.
    try:
        parent = psutil.Process(int(pid))
        for child in parent.children(recursive=True):
            try:
                child.kill()
            except Exception:
                pass
        parent.kill()
        parent.wait(timeout=10)
        print("Killed old process tree:", pid)
    except Exception:
        pass

# Stop old PID from previous run.
if SERVER_PID_PATH.exists():
    old_pid = SERVER_PID_PATH.read_text().strip()
    if old_pid:
        kill_process_tree(old_pid)

# Kill any leftover vLLM server process.
for p in psutil.process_iter(["pid", "name", "cmdline"]):
    try:
        cmdline = " ".join(p.info.get("cmdline") or [])
        if "vllm" in cmdline and "serve" in cmdline:
            kill_process_tree(p.info["pid"])
            print("Killed leftover vLLM process:", p.info["pid"])
    except Exception:
        pass

time.sleep(3)

cmd = [
    "vllm", "serve", MODEL_NAME,

    "--host", "0.0.0.0",
    "--port", str(PORT),

    "--max-model-len", str(MAX_MODEL_LEN),
    "--gpu-memory-utilization", str(GPU_MEMORY_UTILIZATION),

    # Text-only mode.
    "--language-model-only",

    # Qwen3 non-thinking mode at server level.
    "--reasoning-parser", "qwen3",
    "--default-chat-template-kwargs", '{"enable_thinking": false}',

    # Higher concurrency for throughput.
    "--max-num-seqs", str(MAX_NUM_SEQS),
    "--max-num-batched-tokens", str(MAX_NUM_BATCHED_TOKENS),

    # Prefix caching should help because the prompt prefix is mostly shared.
    "--enable-prefix-caching",

    # Use vLLM default generation config, not HF generation_config.json.
    "--generation-config", "vllm",

    # Explicit dtype for Blackwell.
    "--dtype", "bfloat16",

    # Safe for HF repos that need model code.
    "--trust-remote-code",
]

# Do NOT add --enforce-eager.
# Keeping torch.compile / CUDA graph path enabled should improve throughput after startup.

server_env = os.environ.copy()

# Keep this fix: FlashInfer sampler crashes in this Blackwell environment.
server_env["VLLM_USE_FLASHINFER_SAMPLER"] = "0"

# CUDA 13.0 / Blackwell runtime hints.
server_env["VLLM_MAIN_CUDA_VERSION"] = "13.0"
server_env["TORCH_CUDA_ARCH_LIST"] = "12.0"

print("Command:")
print(" ".join(shlex.quote(x) for x in cmd))

print("\nImportant environment variables:")
for k in ["VLLM_USE_FLASHINFER_SAMPLER", "VLLM_MAIN_CUDA_VERSION", "TORCH_CUDA_ARCH_LIST"]:
    print(f"{k}={server_env.get(k)}")

SERVER_LOG_PATH.write_text("", encoding="utf-8")
log_file = open(SERVER_LOG_PATH, "w", encoding="utf-8")

proc = subprocess.Popen(
    cmd,
    stdout=log_file,
    stderr=subprocess.STDOUT,
    text=True,
    env=server_env,
)

SERVER_PID_PATH.write_text(str(proc.pid))

print("\nStarted vLLM server.")
print("PID:", proc.pid)
print("Log:", SERVER_LOG_PATH)

Command:
vllm serve Qwen/Qwen3.5-27B --host 0.0.0.0 --port 8000 --max-model-len 12288 --gpu-memory-utilization 0.88 --language-model-only --reasoning-parser qwen3 --default-chat-template-kwargs '{"enable_thinking": false}' --max-num-seqs 8 --max-num-batched-tokens 16384 --enable-prefix-caching --generation-config vllm --dtype bfloat16 --trust-remote-code

Important environment variables:
VLLM_USE_FLASHINFER_SAMPLER=0
VLLM_MAIN_CUDA_VERSION=13.0
TORCH_CUDA_ARCH_LIST=12.0

Started vLLM server.
PID: 2367
Log: /content/vllm_server.log


In [5]:
# 5) Wait for vLLM server with compact logging
import time
import requests
from pathlib import Path

def tail_log(path, n=80):
    path = Path(path)
    if not path.exists():
        return ""
    lines = path.read_text(errors="ignore").splitlines()
    return "\n".join(lines[-n:])

ready = False

MAX_WAIT_SEC = 1800
SLEEP_SEC = 5
PRINT_EVERY_SEC = 60

start = time.perf_counter()
last_print = -PRINT_EVERY_SEC

for step in range(MAX_WAIT_SEC // SLEEP_SEC):
    elapsed = int(time.perf_counter() - start)

    return_code = proc.poll()
    if return_code is not None:
        print(f"vLLM process exited. Return code: {return_code}")
        print("\n=== Last vLLM log lines ===")
        print(tail_log(SERVER_LOG_PATH, n=160))
        raise RuntimeError("vLLM server crashed or exited during startup.")

    try:
        h = requests.get(f"http://localhost:{PORT}/health", timeout=5)
        if h.status_code == 200:
            m = requests.get(f"{BASE_URL}/models", timeout=10)
            if m.status_code == 200:
                ready = True
                model_info = m.json()["data"][0]
                print("vLLM server is ready.")
                print("Model:", model_info["id"])
                print("Max model len:", model_info.get("max_model_len"))
                break
    except Exception:
        pass

    if elapsed - last_print >= PRINT_EVERY_SEC:
        last_print = elapsed
        print(f"Waiting... {elapsed}s")
        recent = tail_log(SERVER_LOG_PATH, n=12)
        if recent.strip():
            print(recent)
        print("-" * 80)

    time.sleep(SLEEP_SEC)

if not ready:
    print("\n=== Last vLLM log lines ===")
    print(tail_log(SERVER_LOG_PATH, n=160))
    raise RuntimeError("vLLM server did not become ready before timeout.")

Waiting... 0s
--------------------------------------------------------------------------------
Waiting... 60s
(EngineCore pid=2914) INFO 05-21 08:45:33 [registry.py:134] All limits of multimodal modalities supported by the model are set to 0, running in text-only mode.
(EngineCore pid=2914) INFO 05-21 08:45:33 [parallel_state.py:1416] world_size=1 rank=0 local_rank=0 distributed_init_method=tcp://172.28.0.12:36901 backend=nccl
(EngineCore pid=2914) INFO 05-21 08:45:33 [parallel_state.py:1729] rank 0 in world size 1 is assigned as DP rank 0, PP rank 0, PCP rank 0, TP rank 0, EP rank N/A, EPLB rank N/A
(EngineCore pid=2914) INFO 05-21 08:45:33 [topk_topp_sampler.py:70] FlashInfer top-p/top-k sampling disabled via VLLM_USE_FLASHINFER_SAMPLER=0; using PyTorch-native sampler.
(EngineCore pid=2914) INFO 05-21 08:45:33 [gpu_model_runner.py:5006] Starting to load model Qwen/Qwen3.5-27B...
(EngineCore pid=2914) INFO 05-21 08:45:33 [cuda.py:433] Using backend AttentionBackendEnum.FLASH_ATTN for 

In [6]:
# 6) Load all chunks and validate chunk ids
import json
from pathlib import Path

with open(LOCAL_INPUT_PATH, "r", encoding="utf-8") as f:
    chunks = json.load(f)

assert isinstance(chunks, list), "The input JSON must be a list of chunks."

def expected_chunk_id(input_index):
    return f"hotpotqa_chunk_{input_index + 1:08d}"

chunk_id_mismatches = []

for i, chunk in enumerate(chunks):
    got = chunk.get("Chunk_id")
    expected = expected_chunk_id(i)

    if got != expected:
        chunk_id_mismatches.append({
            "input_index": i,
            "expected_chunk_id": expected,
            "got_chunk_id": got,
        })

print("Total chunks:", len(chunks))
print("First chunk:", chunks[0]["Chunk_id"])
print("Last chunk:", chunks[-1]["Chunk_id"])

if chunk_id_mismatches:
    print("Chunk id mismatches:", len(chunk_id_mismatches))
    print(json.dumps(chunk_id_mismatches[:10], ensure_ascii=False, indent=2))
    raise RuntimeError("Chunk_id order does not match input_index order.")

print("Chunk id order is valid.")
print(json.dumps(chunks[0], ensure_ascii=False, indent=2)[:2000])

Total chunks: 35029
First chunk: hotpotqa_chunk_00000001
Last chunk: hotpotqa_chunk_00035029
Chunk id order is valid.
{
  "Chunk_id": "hotpotqa_chunk_00000001",
  "Title": "Meet Corliss Archer",
  "Paragraph_id": [
    1,
    2,
    3,
    4,
    5
  ],
  "Text": "Meet Corliss Archer, a program from radio's Golden Age, ran from January 7, 1943 to September 30, 1956. Although it was CBS's answer to NBC's popular \"A Date with Judy\", it was also broadcast by NBC in 1948 as a summer replacement for \"The Bob Hope Show\". From October 3, 1952 to June 26, 1953, it aired on ABC, finally returning to CBS. Despite the program's long run, fewer than 24 episodes are known to exist.\nPriscilla Lyon and Janet Waldo successively portrayed 15-year-old Corliss on radio. Lugene Sanders also played Corliss briefly on radio and in the CBS version of the \"Meet Corliss Archer\" television show.\nPerpetually perky, breathless and well-intentioned, Corliss is constantly at the side of her next-door neighb

In [7]:
# 7) Define schema and prompt
import json

KG_SCHEMA = {
    "type": "object",
    "additionalProperties": False,
    "properties": {
        "entities": {
            "type": "array",
            "maxItems": 30,
            "items": {
                "type": "string",
                "minLength": 1
            }
        },
        "relations": {
            "type": "array",
            "maxItems": 35,
            "items": {
                "type": "object",
                "additionalProperties": False,
                "properties": {
                    "head": {
                        "type": "string",
                        "minLength": 1
                    },
                    "relation": {
                        "type": "string",
                        "minLength": 1
                    },
                    "tail": {
                        "type": "string",
                        "minLength": 1
                    }
                },
                "required": ["head", "relation", "tail"]
            }
        },
        "facts": {
            "type": "array",
            "maxItems": 30,
            "items": {
                "type": "object",
                "additionalProperties": False,
                "properties": {
                    "entity": {
                        "type": "string",
                        "minLength": 1
                    },
                    "info": {
                        "type": "string",
                        "minLength": 1
                    }
                },
                "required": ["entity", "info"]
            }
        }
    },
    "required": ["entities", "relations", "facts"]
}

EXAMPLE_OUTPUT = {
    "entities": [
        "Marie Curie",
        "radioactivity",
        "Pierre Curie",
        "Curie Institute",
        "Paris",
        "1920"
    ],
    "relations": [
        {
            "head": "Marie Curie",
            "relation": "Marie Curie conducted pioneering research on radioactivity.",
            "tail": "radioactivity"
        },
        {
            "head": "Marie Curie",
            "relation": "Marie Curie was married to Pierre Curie.",
            "tail": "Pierre Curie"
        },
        {
            "head": "Curie Institute",
            "relation": "The Curie Institute was founded in Paris.",
            "tail": "Paris"
        },
        {
            "head": "Curie Institute",
            "relation": "The Curie Institute was founded in 1920.",
            "tail": "1920"
        }
    ],
    "facts": [
        {
            "entity": "Marie Curie",
            "info": "Marie Curie was a Polish and naturalized-French physicist and chemist who researched radioactivity."
        },
        {
            "entity": "Pierre Curie",
            "info": "Pierre Curie was married to Marie Curie."
        },
        {
            "entity": "Curie Institute",
            "info": "The Curie Institute in Paris was founded in 1920."
        }
    ]
}

def build_prompt(chunk):
    chunk_id = chunk["Chunk_id"]
    title = chunk["Title"]
    paragraph_ids = json.dumps(chunk["Paragraph_id"], ensure_ascii=False)
    chunk_text = chunk["Text"]

    return f"""You are an expert information extraction system designed to build a highly accurate retrieval knowledge graph.

Your task is to extract entities, relations between entities, and specific facts about entities from a given Wikipedia chunk.

Extract only facts explicitly supported by the chunk.
Do not use external knowledge.
Do not infer facts that are not clearly stated.
Prefer precision over recall.
Return JSON only.

Extract:

1. entities:
Important specific entities useful for multi-hop retrieval.
Include specific people, organizations, locations, works, events, awards, dates/years, and key concepts.
Do not extract generic adjectives or common nouns as standalone entities.
Extract at most 30 entities.

2. relations:
A relation is a connection between two extracted entities that is explicitly stated or directly supported by the chunk.
Each relation must have:
- head: one entity copied exactly from entities
- tail: one entity copied exactly from entities
- relation: a short, simple natural-language sentence explaining the connection between head and tail
Extract at most 35 relations.

3. facts:
A fact is a short, simple natural-language sentence describing what specific information this exact chunk provides about one extracted entity.
Each fact must have:
- entity: one entity copied exactly from entities
- info: a short sentence about that entity based only on this chunk
Extract at most 30 facts.

Rules:
- Every head and tail in relations must be copied exactly from the entities array.
- Every entity in facts must be copied exactly from the entities array.
- Do not create duplicate entities, duplicate relations, or duplicate facts.
- Resolve pronouns to actual entity names only when unambiguous.
- Generic words may appear inside relation and info sentences, but not as standalone entities.
- The output must be valid JSON strictly matching the required schema.

Example:

Input:
chunk_id:
ex_001

title:
Marie Curie

paragraph_ids:
[1]

text:
Marie Curie was a Polish and naturalized-French physicist and chemist who conducted pioneering research on radioactivity. She was married to Pierre Curie. The Curie Institute in Paris was founded in 1920.

Output:
{json.dumps(EXAMPLE_OUTPUT, ensure_ascii=False, indent=2)}

Now extract from this chunk.

Input:
chunk_id:
{chunk_id}

title:
{title}

paragraph_ids:
{paragraph_ids}

text:
{chunk_text}

Output:
"""

In [8]:
# 8) Define client and adaptive extraction helpers
import json
import time
from openai import OpenAI

client = OpenAI(
    api_key="EMPTY",
    base_url=BASE_URL,
    timeout=3600,
)

# Adaptive output budget.
OUTPUT_TOKEN_STEPS = [4096, 8192]
MAX_OUTPUT_TOKENS = max(OUTPUT_TOKEN_STEPS)

# Qwen3.5 non-thinking general-task parameters.
TEMPERATURE = 0.7
TOP_P = 0.8
TOP_K = 20
MIN_P = 0.0

# For information extraction, 0.0 is safer than 1.5.
PRESENCE_PENALTY = 0.0
REPETITION_PENALTY = 1.0

SAVE_RAW_RESPONSE_ON_ERROR = True

def make_messages(chunk):
    return [
        {
            "role": "system",
            "content": "You extract knowledge graph data from Wikipedia chunks. Return valid JSON only."
        },
        {
            "role": "user",
            "content": build_prompt(chunk)
        }
    ]

def usage_to_dict(usage):
    if usage is None:
        return None

    return {
        "prompt_tokens": getattr(usage, "prompt_tokens", None),
        "completion_tokens": getattr(usage, "completion_tokens", None),
        "total_tokens": getattr(usage, "total_tokens", None),
    }

def call_llm_once(chunk, max_tokens):
    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=make_messages(chunk),
        max_tokens=max_tokens,
        temperature=TEMPERATURE,
        top_p=TOP_P,
        presence_penalty=PRESENCE_PENALTY,
        seed=42,
        extra_body={
            "top_k": TOP_K,
            "min_p": MIN_P,
            "repetition_penalty": REPETITION_PENALTY,
            "chat_template_kwargs": {
                "enable_thinking": False
            },
            "structured_outputs": {
                "json": KG_SCHEMA
            }
        },
    )

    choice = response.choices[0]

    return {
        "content": choice.message.content,
        "finish_reason": choice.finish_reason,
        "usage": usage_to_dict(response.usage),
        "max_tokens": max_tokens,
    }

def extract_one_raw_adaptive(chunk, max_retries_per_budget=1):
    # Store exactly the model JSON fields after JSON parsing.
    start = time.perf_counter()
    last_error = None
    raw = None
    last_finish_reason = None
    last_usage = None
    last_max_tokens = None
    attempts = []

    for max_tokens in OUTPUT_TOKEN_STEPS:
        for retry_id in range(max_retries_per_budget + 1):
            try:
                response_data = call_llm_once(chunk, max_tokens=max_tokens)

                raw = response_data["content"]
                finish_reason = response_data["finish_reason"]
                usage = response_data["usage"]

                last_finish_reason = finish_reason
                last_usage = usage
                last_max_tokens = max_tokens

                attempts.append({
                    "max_tokens": max_tokens,
                    "retry_id": retry_id,
                    "finish_reason": finish_reason,
                    "usage": usage,
                })

                # If the model hit the token budget, retry with a larger budget.
                if finish_reason == "length":
                    last_error = f"finish_reason=length at max_tokens={max_tokens}"
                    break

                # Parse only to make final saved file valid JSON.
                model_output = json.loads(raw)

                return {
                    "chunk_id": chunk["Chunk_id"],
                    "title": chunk["Title"],
                    "paragraph_id": chunk["Paragraph_id"],
                    "token_count": chunk.get("Token_count"),

                    # Model output copied as-is after JSON parsing.
                    "entities": model_output.get("entities"),
                    "relations": model_output.get("relations"),
                    "facts": model_output.get("facts"),

                    "error": None,
                    "latency_sec": round(time.perf_counter() - start, 3),
                    "finish_reason": finish_reason,
                    "max_tokens_used": max_tokens,
                    "usage": usage,
                    "attempts": attempts,
                }

            except Exception as e:
                last_error = repr(e)
                time.sleep(1.0 * (retry_id + 1))

        # Continue to the next larger max_tokens budget.

    failed_item = {
        "chunk_id": chunk["Chunk_id"],
        "title": chunk["Title"],
        "paragraph_id": chunk["Paragraph_id"],
        "token_count": chunk.get("Token_count"),
        "entities": None,
        "relations": None,
        "facts": None,
        "error": last_error,
        "latency_sec": round(time.perf_counter() - start, 3),
        "finish_reason": last_finish_reason,
        "max_tokens_used": last_max_tokens,
        "usage": last_usage,
        "attempts": attempts,
    }

    if SAVE_RAW_RESPONSE_ON_ERROR:
        failed_item["raw_response"] = raw

    return failed_item

In [9]:
# 9) Audit existing KG files and find missing or failed indices
import json
import os
import re
from pathlib import Path
from collections import defaultdict, Counter
from datetime import datetime, timezone

REPROCESS_ERROR_ITEMS = True

FINAL_OUTPUT_RE = re.compile(
    r"^hotpotqa_kg_extractions_(?:first\d+_)?chunks_\d{8}_to_\d{8}\.json$"
)

AUDIT_REPORT_PATH = KG_DIR / "hotpotqa_kg_audit_report_before_repair.json"
REPAIR_INDEX_PATH = KG_DIR / "hotpotqa_missing_or_failed_indices.json"

def atomic_json_dump(obj, path):
    # Write safely, then replace.
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp_path = path.with_name(path.name + ".tmp")

    with open(tmp_path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)
        f.flush()
        os.fsync(f.fileno())

    os.replace(tmp_path, path)

def summarize_ranges(values, max_ranges=30):
    # Make compact ranges for printing.
    values = sorted(set(values))

    if not values:
        return []

    ranges = []
    start = prev = values[0]

    for x in values[1:]:
        if x == prev + 1:
            prev = x
        else:
            ranges.append((start, prev))
            start = prev = x

    ranges.append((start, prev))

    text_ranges = [
        str(a) if a == b else f"{a}-{b}"
        for a, b in ranges[:max_ranges]
    ]

    if len(ranges) > max_ranges:
        text_ranges.append(f"... plus {len(ranges) - max_ranges} more ranges")

    return text_ranges

def parse_range_from_filename(path):
    # Parse chunk-number range from filename.
    m = re.search(r"chunks_(\d{8})_to_(\d{8})\.json$", path.name)

    if not m:
        return None

    return {
        "first_chunk_number": int(m.group(1)),
        "last_chunk_number": int(m.group(2)),
        "start_index": int(m.group(1)) - 1,
        "end_index_exclusive": int(m.group(2)),
    }

final_paths = sorted(
    p for p in KG_DIR.glob("hotpotqa_kg_extractions_*.json")
    if FINAL_OUTPUT_RE.match(p.name)
)

assert final_paths, f"No final extraction files found in: {KG_DIR}"

records_by_index = defaultdict(list)
file_summaries = []
invalid_records = []
mismatched_records = []

for path in final_paths:
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)

    if not isinstance(data, list):
        invalid_records.append({
            "file": str(path),
            "reason": "File content is not a list.",
        })
        continue

    parsed_range = parse_range_from_filename(path)
    indices_in_file = []
    errors_in_file = 0

    for local_pos, item in enumerate(data):
        if not isinstance(item, dict):
            invalid_records.append({
                "file": str(path),
                "local_pos": local_pos,
                "reason": "Item is not a dict.",
            })
            continue

        input_index = item.get("input_index")

        if input_index is None:
            chunk_id = item.get("chunk_id")

            if isinstance(chunk_id, str):
                m = re.search(r"hotpotqa_chunk_(\d{8})$", chunk_id)
                if m:
                    input_index = int(m.group(1)) - 1

        if not isinstance(input_index, int):
            invalid_records.append({
                "file": str(path),
                "local_pos": local_pos,
                "chunk_id": item.get("chunk_id"),
                "reason": "Missing or invalid input_index.",
            })
            continue

        if input_index < 0 or input_index >= len(chunks):
            invalid_records.append({
                "file": str(path),
                "local_pos": local_pos,
                "input_index": input_index,
                "chunk_id": item.get("chunk_id"),
                "reason": "input_index out of range.",
            })
            continue

        expected_id = chunks[input_index]["Chunk_id"]
        got_id = item.get("chunk_id")

        if got_id != expected_id:
            mismatched_records.append({
                "file": str(path),
                "local_pos": local_pos,
                "input_index": input_index,
                "expected_chunk_id": expected_id,
                "got_chunk_id": got_id,
            })

        if item.get("error") is not None:
            errors_in_file += 1

        indices_in_file.append(input_index)

        records_by_index[input_index].append({
            "file": str(path),
            "local_pos": local_pos,
            "chunk_id": got_id,
            "error": item.get("error"),
        })

    file_summaries.append({
        "file": str(path),
        "filename": path.name,
        "declared_range": parsed_range,
        "num_items": len(data),
        "min_input_index": min(indices_in_file) if indices_in_file else None,
        "max_input_index": max(indices_in_file) if indices_in_file else None,
        "num_unique_input_indices": len(set(indices_in_file)),
        "num_errors": errors_in_file,
    })

all_indices = set(range(len(chunks)))
covered_indices = set(records_by_index.keys())
missing_input_indices = sorted(all_indices - covered_indices)

duplicate_input_indices = sorted(
    idx for idx, rows in records_by_index.items()
    if len(rows) > 1
)

error_input_indices = sorted(
    idx for idx, rows in records_by_index.items()
    if any(row.get("error") is not None for row in rows)
)

mismatched_input_indices = sorted(
    set(row["input_index"] for row in mismatched_records)
)

repair_input_indices = set(missing_input_indices)
repair_reasons = defaultdict(set)

for idx in missing_input_indices:
    repair_reasons[idx].add("missing")

if REPROCESS_ERROR_ITEMS:
    repair_input_indices.update(error_input_indices)

    for idx in error_input_indices:
        repair_reasons[idx].add("error")

repair_input_indices.update(mismatched_input_indices)

for idx in mismatched_input_indices:
    repair_reasons[idx].add("chunk_id_mismatch")

repair_input_indices = sorted(repair_input_indices)

repair_items = [
    {
        "input_index": idx,
        "chunk_number": idx + 1,
        "chunk_id": chunks[idx]["Chunk_id"],
        "title": chunks[idx].get("Title"),
        "reasons": sorted(repair_reasons[idx]),
    }
    for idx in repair_input_indices
]

audit_report = {
    "dataset": "hotpotqa",
    "input_path": str(GDRIVE_INPUT_PATH),
    "kg_dir": str(KG_DIR),
    "total_chunks": len(chunks),
    "num_final_files": len(final_paths),
    "final_files": file_summaries,
    "num_covered_input_indices": len(covered_indices),
    "num_missing_input_indices": len(missing_input_indices),
    "num_duplicate_input_indices": len(duplicate_input_indices),
    "num_error_input_indices": len(error_input_indices),
    "num_mismatched_records": len(mismatched_records),
    "reprocess_error_items": REPROCESS_ERROR_ITEMS,
    "num_repair_input_indices": len(repair_input_indices),
    "missing_input_index_ranges_0_based": summarize_ranges(missing_input_indices),
    "missing_chunk_number_ranges_1_based": summarize_ranges([i + 1 for i in missing_input_indices]),
    "error_input_index_ranges_0_based": summarize_ranges(error_input_indices),
    "repair_input_index_ranges_0_based": summarize_ranges(repair_input_indices),
    "repair_chunk_number_ranges_1_based": summarize_ranges([i + 1 for i in repair_input_indices]),
    "duplicate_input_indices": duplicate_input_indices,
    "invalid_records": invalid_records,
    "mismatched_records": mismatched_records,
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
}

atomic_json_dump(audit_report, AUDIT_REPORT_PATH)
atomic_json_dump(repair_items, REPAIR_INDEX_PATH)

print("Final files found:", len(final_paths))
for s in file_summaries:
    print(
        s["filename"],
        "| items:", s["num_items"],
        "| unique:", s["num_unique_input_indices"],
        "| errors:", s["num_errors"],
        "| min/max input_index:", s["min_input_index"], s["max_input_index"],
    )

print("\nCovered input indices:", len(covered_indices), "/", len(chunks))
print("Missing input indices:", len(missing_input_indices))
print("Missing chunk numbers:", summarize_ranges([i + 1 for i in missing_input_indices]))
print("Duplicate input indices:", len(duplicate_input_indices))
print("Error input indices:", len(error_input_indices))
print("Mismatched records:", len(mismatched_records))
print("Repair input indices:", len(repair_input_indices))
print("Repair chunk numbers:", summarize_ranges([i + 1 for i in repair_input_indices]))

print("\nSaved audit report:", AUDIT_REPORT_PATH)
print("Saved repair index list:", REPAIR_INDEX_PATH)

Final files found: 7
hotpotqa_kg_extractions_chunks_00005001_to_00010000.json | items: 5000 | unique: 5000 | errors: 6 | min/max input_index: 5000 9999
hotpotqa_kg_extractions_chunks_00010001_to_00015000.json | items: 5000 | unique: 5000 | errors: 3 | min/max input_index: 10000 14999
hotpotqa_kg_extractions_chunks_00015001_to_00020000.json | items: 5000 | unique: 5000 | errors: 3 | min/max input_index: 15000 19999
hotpotqa_kg_extractions_chunks_00020001_to_00025000.json | items: 5000 | unique: 5000 | errors: 3 | min/max input_index: 20000 24999
hotpotqa_kg_extractions_chunks_00025001_to_00030000.json | items: 5000 | unique: 5000 | errors: 2 | min/max input_index: 25000 29999
hotpotqa_kg_extractions_chunks_00030001_to_00035000.json | items: 5000 | unique: 5000 | errors: 0 | min/max input_index: 30000 34999
hotpotqa_kg_extractions_first5000_chunks_00000001_to_00005000.json | items: 5000 | unique: 5000 | errors: 2 | min/max input_index: 0 4999

Covered input indices: 35000 / 35029
Missing

In [10]:
# 10) Run KG extraction only for missing or failed chunks
from tqdm.auto import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed
import json
import time
import os
from datetime import datetime, timezone

REPAIR_OUTPUT_PATH = KG_DIR / "hotpotqa_kg_extractions_repair_missing_failed.json"
REPAIR_META_PATH = KG_DIR / "hotpotqa_kg_extractions_repair_missing_failed_meta.json"
REPAIR_PARTIAL_PATH = KG_DIR / "hotpotqa_kg_extractions_repair_missing_failed.partial.json"

MAX_WORKERS = 8
SAVE_EVERY = 10
RETRY_FAILED_REPAIR_ITEMS = True

repair_indices = repair_input_indices

print("Repair items to process:", len(repair_indices))
print("Repair chunk numbers:", summarize_ranges([i + 1 for i in repair_indices]))

def load_existing_repair_items():
    # Prefer final repair output, then repair checkpoint.
    for path in [REPAIR_OUTPUT_PATH, REPAIR_PARTIAL_PATH]:
        if path.exists():
            try:
                with open(path, "r", encoding="utf-8") as f:
                    data = json.load(f)

                if isinstance(data, list):
                    return data
            except Exception:
                pass

    return []

def make_repair_failed_item(global_index, chunk, error):
    # Store failures without stopping the repair run.
    return {
        "chunk_id": chunk["Chunk_id"],
        "title": chunk["Title"],
        "paragraph_id": chunk["Paragraph_id"],
        "token_count": chunk.get("Token_count"),
        "entities": None,
        "relations": None,
        "facts": None,
        "error": error,
        "latency_sec": None,
        "finish_reason": None,
        "max_tokens_used": None,
        "usage": None,
        "attempts": [],
        "input_index": global_index,
        "repair_local_index": repair_indices.index(global_index),
        "repair_run": True,
    }

def save_repair_partial(results_by_index):
    # Save completed repair items only.
    partial_results = [
        results_by_index[idx]
        for idx in sorted(results_by_index)
    ]
    atomic_json_dump(partial_results, REPAIR_PARTIAL_PATH)

results_by_index = {}

existing_repair_items = load_existing_repair_items()

for item in existing_repair_items:
    idx = item.get("input_index")

    if idx not in repair_indices:
        continue

    if RETRY_FAILED_REPAIR_ITEMS and item.get("error") is not None:
        continue

    results_by_index[idx] = item

pending_jobs = [
    (idx, chunks[idx])
    for idx in repair_indices
    if idx not in results_by_index
]

print("Already completed repair items:", len(results_by_index))
print("Pending repair items:", len(pending_jobs))

def run_one_repair(index_and_chunk):
    global_index, chunk = index_and_chunk

    item = extract_one_raw_adaptive(chunk, max_retries_per_budget=1)

    item["input_index"] = global_index
    item["repair_local_index"] = repair_indices.index(global_index)
    item["repair_run"] = True

    return global_index, item

overall_start = time.perf_counter()
completed_new = 0
executor = None
cancelled = False

if pending_jobs:
    try:
        executor = ThreadPoolExecutor(max_workers=MAX_WORKERS)

        futures = {
            executor.submit(run_one_repair, job): job
            for job in pending_jobs
        }

        for future in tqdm(
            as_completed(futures),
            total=len(futures),
            desc="Repairing missing or failed KG items"
        ):
            global_index, chunk = futures[future]

            try:
                _, item = future.result()
            except Exception as e:
                item = make_repair_failed_item(global_index, chunk, repr(e))

            results_by_index[global_index] = item
            completed_new += 1

            if completed_new % SAVE_EVERY == 0:
                save_repair_partial(results_by_index)

    except KeyboardInterrupt:
        cancelled = True
        save_repair_partial(results_by_index)

        if executor is not None:
            executor.shutdown(wait=False, cancel_futures=True)

        raise

    finally:
        save_repair_partial(results_by_index)

        if executor is not None and not cancelled:
            executor.shutdown(wait=True)

missing_after_repair_run = [
    idx for idx in repair_indices
    if idx not in results_by_index
]

if missing_after_repair_run:
    raise RuntimeError(
        f"{len(missing_after_repair_run)} repair items are still missing. "
        f"Re-run this cell to resume from: {REPAIR_PARTIAL_PATH}"
    )

repair_results = [
    results_by_index[idx]
    for idx in sorted(results_by_index)
]

total_time_sec = time.perf_counter() - overall_start

num_errors = sum(1 for x in repair_results if x.get("error") is not None)
latencies = [
    x["latency_sec"]
    for x in repair_results
    if x.get("latency_sec") is not None
]

token_budget_counts = Counter(str(x.get("max_tokens_used")) for x in repair_results)
finish_reason_counts = Counter(str(x.get("finish_reason")) for x in repair_results)

repair_meta = {
    "dataset": "hotpotqa",
    "input_path": str(GDRIVE_INPUT_PATH),
    "repair_index_path": str(REPAIR_INDEX_PATH),
    "output_path": str(REPAIR_OUTPUT_PATH),
    "partial_path": str(REPAIR_PARTIAL_PATH),
    "model": MODEL_NAME,
    "num_repair_items": len(repair_results),
    "num_errors": num_errors,
    "repair_input_index_ranges_0_based": summarize_ranges([x["input_index"] for x in repair_results]),
    "repair_chunk_number_ranges_1_based": summarize_ranges([x["input_index"] + 1 for x in repair_results]),
    "max_workers": MAX_WORKERS,
    "save_every": SAVE_EVERY,
    "output_token_steps": OUTPUT_TOKEN_STEPS,
    "temperature": TEMPERATURE,
    "top_p": TOP_P,
    "top_k": TOP_K,
    "min_p": MIN_P,
    "presence_penalty": PRESENCE_PENALTY,
    "repetition_penalty": REPETITION_PENALTY,
    "token_budget_counts": dict(token_budget_counts),
    "finish_reason_counts": dict(finish_reason_counts),
    "total_time_sec": round(total_time_sec, 3),
    "avg_latency_sec": round(sum(latencies) / len(latencies), 3) if latencies else None,
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
}

atomic_json_dump(repair_results, REPAIR_OUTPUT_PATH)
atomic_json_dump(repair_meta, REPAIR_META_PATH)
save_repair_partial(results_by_index)

print("Saved repair JSON:", REPAIR_OUTPUT_PATH)
print("Saved repair checkpoint JSON:", REPAIR_PARTIAL_PATH)
print("Saved repair meta JSON:", REPAIR_META_PATH)
print("Repair errors:", num_errors)

Repair items to process: 48
Repair chunk numbers: ['2260', '3182', '5449', '5797', '6516', '6947-6948', '7536', '10286', '12953', '13113', '16563', '19278', '19603', '20332', '21074', '24680', '25572', '29833', '35001-35029']
Already completed repair items: 0
Pending repair items: 48


Repairing missing or failed KG items:   0%|          | 0/48 [00:00<?, ?it/s]

Saved repair JSON: /content/drive/MyDrive/final_project/idea_1/kg/hotpotqa_kg_extractions_repair_missing_failed.json
Saved repair checkpoint JSON: /content/drive/MyDrive/final_project/idea_1/kg/hotpotqa_kg_extractions_repair_missing_failed.partial.json
Saved repair meta JSON: /content/drive/MyDrive/final_project/idea_1/kg/hotpotqa_kg_extractions_repair_missing_failed_meta.json
Repair errors: 16


In [11]:
# 11) Final audit after adding repair file
import json
from datetime import datetime, timezone

FINAL_AUDIT_REPORT_PATH = KG_DIR / "hotpotqa_kg_audit_report_after_repair.json"

combined_paths = final_paths.copy()

if REPAIR_OUTPUT_PATH.exists():
    combined_paths.append(REPAIR_OUTPUT_PATH)

final_records_by_index = defaultdict(list)
final_invalid_records = []
final_mismatched_records = []

for path in combined_paths:
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)

    if not isinstance(data, list):
        final_invalid_records.append({
            "file": str(path),
            "reason": "File content is not a list.",
        })
        continue

    for local_pos, item in enumerate(data):
        if not isinstance(item, dict):
            final_invalid_records.append({
                "file": str(path),
                "local_pos": local_pos,
                "reason": "Item is not a dict.",
            })
            continue

        idx = item.get("input_index")

        if not isinstance(idx, int) or idx < 0 or idx >= len(chunks):
            final_invalid_records.append({
                "file": str(path),
                "local_pos": local_pos,
                "input_index": idx,
                "chunk_id": item.get("chunk_id"),
                "reason": "Invalid input_index.",
            })
            continue

        expected_id = chunks[idx]["Chunk_id"]
        got_id = item.get("chunk_id")

        if got_id != expected_id:
            final_mismatched_records.append({
                "file": str(path),
                "local_pos": local_pos,
                "input_index": idx,
                "expected_chunk_id": expected_id,
                "got_chunk_id": got_id,
            })

        final_records_by_index[idx].append({
            "file": str(path),
            "local_pos": local_pos,
            "chunk_id": got_id,
            "error": item.get("error"),
        })

final_covered_indices = set(final_records_by_index.keys())
final_missing_input_indices = sorted(set(range(len(chunks))) - final_covered_indices)

final_error_input_indices = sorted(
    idx for idx, rows in final_records_by_index.items()
    if any(row.get("error") is not None for row in rows)
)

final_duplicate_input_indices = sorted(
    idx for idx, rows in final_records_by_index.items()
    if len(rows) > 1
)

final_audit_report = {
    "dataset": "hotpotqa",
    "total_chunks": len(chunks),
    "num_files_checked": len(combined_paths),
    "files_checked": [str(p) for p in combined_paths],
    "num_covered_input_indices": len(final_covered_indices),
    "num_missing_input_indices": len(final_missing_input_indices),
    "num_error_input_indices": len(final_error_input_indices),
    "num_duplicate_input_indices": len(final_duplicate_input_indices),
    "num_invalid_records": len(final_invalid_records),
    "num_mismatched_records": len(final_mismatched_records),
    "missing_input_index_ranges_0_based": summarize_ranges(final_missing_input_indices),
    "missing_chunk_number_ranges_1_based": summarize_ranges([i + 1 for i in final_missing_input_indices]),
    "error_input_index_ranges_0_based": summarize_ranges(final_error_input_indices),
    "error_chunk_number_ranges_1_based": summarize_ranges([i + 1 for i in final_error_input_indices]),
    "duplicate_input_indices": final_duplicate_input_indices,
    "invalid_records": final_invalid_records,
    "mismatched_records": final_mismatched_records,
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
}

atomic_json_dump(final_audit_report, FINAL_AUDIT_REPORT_PATH)

print("Final covered input indices:", len(final_covered_indices), "/", len(chunks))
print("Final missing input indices:", len(final_missing_input_indices))
print("Final missing chunk numbers:", summarize_ranges([i + 1 for i in final_missing_input_indices]))
print("Final error input indices:", len(final_error_input_indices))
print("Final error chunk numbers:", summarize_ranges([i + 1 for i in final_error_input_indices]))
print("Final duplicate input indices:", len(final_duplicate_input_indices))
print("Final mismatched records:", len(final_mismatched_records))
print("Saved final audit report:", FINAL_AUDIT_REPORT_PATH)

if final_missing_input_indices:
    raise RuntimeError("Some chunks are still missing. Check the final audit report.")

if final_error_input_indices:
    print("Warning: Some chunks still have error != None. Check the final audit report.")
else:
    print("All chunks are covered and no error items remain.")

Final covered input indices: 35029 / 35029
Final missing input indices: 0
Final missing chunk numbers: []
Final error input indices: 19
Final error chunk numbers: ['2260', '3182', '5449', '5797', '6516', '6947-6948', '7536', '10286', '12953', '13113', '16563', '19278', '19603', '20332', '21074', '24680', '25572', '29833']
Final duplicate input indices: 19
Final mismatched records: 0
Saved final audit report: /content/drive/MyDrive/final_project/idea_1/kg/hotpotqa_kg_audit_report_after_repair.json


In [12]:
# 11) Safe retry + clean merged final file
import json
import os
import re
import copy
import time
from pathlib import Path
from collections import defaultdict, Counter
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime, timezone
from tqdm.auto import tqdm

# -----------------------------
# Paths
# -----------------------------
ORIGINAL_FINAL_RE = re.compile(
    r"^hotpotqa_kg_extractions_(?:first\d+_)?chunks_\d{8}_to_\d{8}\.json$"
)

REPAIR_OUTPUT_PATH = KG_DIR / "hotpotqa_kg_extractions_repair_missing_failed.json"
REPAIR_PARTIAL_PATH = KG_DIR / "hotpotqa_kg_extractions_repair_missing_failed.partial.json"

SAFE_REPAIR_OUTPUT_PATH = KG_DIR / "hotpotqa_kg_extractions_repair_safe_retry.json"
SAFE_REPAIR_PARTIAL_PATH = KG_DIR / "hotpotqa_kg_extractions_repair_safe_retry.partial.json"
SAFE_REPAIR_META_PATH = KG_DIR / "hotpotqa_kg_extractions_repair_safe_retry_meta.json"

CLEAN_OUTPUT_PATH = KG_DIR / f"hotpotqa_kg_extractions_all_00000001_to_{len(chunks):08d}.clean.json"
CLEAN_AUDIT_PATH = KG_DIR / f"hotpotqa_kg_extractions_all_00000001_to_{len(chunks):08d}.clean_audit.json"

RUN_SAFE_RETRY = True
SAFE_MAX_WORKERS = 4
SAFE_SAVE_EVERY = 5

# -----------------------------
# Small helpers
# -----------------------------
def atomic_json_dump(obj, path):
    # Write safely, then replace.
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp_path = path.with_name(path.name + ".tmp")

    with open(tmp_path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)
        f.flush()
        os.fsync(f.fileno())

    os.replace(tmp_path, path)

def summarize_ranges(values, max_ranges=30):
    # Compact ranges for printing.
    values = sorted(set(values))

    if not values:
        return []

    ranges = []
    start = prev = values[0]

    for x in values[1:]:
        if x == prev + 1:
            prev = x
        else:
            ranges.append((start, prev))
            start = prev = x

    ranges.append((start, prev))

    out = [
        str(a) if a == b else f"{a}-{b}"
        for a, b in ranges[:max_ranges]
    ]

    if len(ranges) > max_ranges:
        out.append(f"... plus {len(ranges) - max_ranges} more ranges")

    return out

def input_index_from_item(item):
    # Get input_index from item or chunk_id.
    idx = item.get("input_index")

    if isinstance(idx, int):
        return idx

    chunk_id = item.get("chunk_id")

    if isinstance(chunk_id, str):
        m = re.search(r"hotpotqa_chunk_(\d{8})$", chunk_id)
        if m:
            return int(m.group(1)) - 1

    return None

def expected_chunk_id(idx):
    return chunks[idx]["Chunk_id"]

def is_valid_success(item, idx):
    # A usable record must be successful and schema-shaped.
    if not isinstance(item, dict):
        return False

    if idx < 0 or idx >= len(chunks):
        return False

    if item.get("chunk_id") != expected_chunk_id(idx):
        return False

    if item.get("error") is not None:
        return False

    if item.get("finish_reason") == "length":
        return False

    if not isinstance(item.get("entities"), list):
        return False

    if not isinstance(item.get("relations"), list):
        return False

    if not isinstance(item.get("facts"), list):
        return False

    return True

def load_json_list(path):
    if not path.exists():
        return []

    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)

    if not isinstance(data, list):
        raise ValueError(f"Expected list JSON: {path}")

    return data

def add_candidates_from_path(candidates, path, source_kind, source_priority):
    # Add records grouped by input_index.
    data = load_json_list(path)

    bad_items = 0

    for local_pos, item in enumerate(data):
        if not isinstance(item, dict):
            bad_items += 1
            continue

        idx = input_index_from_item(item)

        if not isinstance(idx, int) or idx < 0 or idx >= len(chunks):
            bad_items += 1
            continue

        wrapped = {
            "item": item,
            "path": str(path),
            "filename": path.name,
            "local_pos": local_pos,
            "source_kind": source_kind,
            "source_priority": source_priority,
            "success": is_valid_success(item, idx),
        }

        candidates[idx].append(wrapped)

    return {
        "path": str(path),
        "source_kind": source_kind,
        "num_items": len(data),
        "bad_items": bad_items,
    }

def collect_candidates():
    # Original batch files only.
    candidates = defaultdict(list)
    source_summaries = []

    original_paths = sorted(
        p for p in KG_DIR.glob("hotpotqa_kg_extractions_*.json")
        if ORIGINAL_FINAL_RE.match(p.name)
    )

    for path in original_paths:
        source_summaries.append(
            add_candidates_from_path(
                candidates=candidates,
                path=path,
                source_kind="original_batch",
                source_priority=10,
            )
        )

    # Repair v1: use final if available, else partial.
    if REPAIR_OUTPUT_PATH.exists():
        source_summaries.append(
            add_candidates_from_path(
                candidates=candidates,
                path=REPAIR_OUTPUT_PATH,
                source_kind="repair_v1",
                source_priority=20,
            )
        )
    elif REPAIR_PARTIAL_PATH.exists():
        source_summaries.append(
            add_candidates_from_path(
                candidates=candidates,
                path=REPAIR_PARTIAL_PATH,
                source_kind="repair_v1_partial",
                source_priority=20,
            )
        )

    # Repair v2 safe retry.
    if SAFE_REPAIR_OUTPUT_PATH.exists():
        source_summaries.append(
            add_candidates_from_path(
                candidates=candidates,
                path=SAFE_REPAIR_OUTPUT_PATH,
                source_kind="repair_safe_retry",
                source_priority=30,
            )
        )
    elif SAFE_REPAIR_PARTIAL_PATH.exists():
        source_summaries.append(
            add_candidates_from_path(
                candidates=candidates,
                path=SAFE_REPAIR_PARTIAL_PATH,
                source_kind="repair_safe_retry_partial",
                source_priority=30,
            )
        )

    return candidates, source_summaries

def choose_best_item(rows):
    # Prefer successful records, then newer repair records.
    def score(row):
        item = row["item"]
        success_score = 1 if row["success"] else 0
        source_priority = row["source_priority"]
        finish_not_length = 1 if item.get("finish_reason") != "length" else 0
        has_any_output = int(
            isinstance(item.get("entities"), list)
            or isinstance(item.get("relations"), list)
            or isinstance(item.get("facts"), list)
        )

        return (
            success_score,
            source_priority,
            finish_not_length,
            has_any_output,
            -row["local_pos"],
        )

    return max(rows, key=score)

def build_clean_from_candidates():
    # Build one best record per input_index.
    candidates, source_summaries = collect_candidates()

    clean_items_by_index = {}
    missing_indices = []
    duplicate_source_indices = []
    true_error_indices = []
    chosen_sources = Counter()

    for idx in range(len(chunks)):
        rows = candidates.get(idx, [])

        if not rows:
            missing_indices.append(idx)
            continue

        if len(rows) > 1:
            duplicate_source_indices.append(idx)

        best = choose_best_item(rows)
        chosen = copy.deepcopy(best["item"])

        chosen["input_index"] = idx
        chosen["final_clean_source_file"] = best["filename"]
        chosen["final_clean_source_kind"] = best["source_kind"]

        clean_items_by_index[idx] = chosen
        chosen_sources[best["source_kind"]] += 1

        if not is_valid_success(chosen, idx):
            true_error_indices.append(idx)

    clean_items = [
        clean_items_by_index[idx]
        for idx in sorted(clean_items_by_index)
    ]

    return {
        "clean_items": clean_items,
        "clean_items_by_index": clean_items_by_index,
        "missing_indices": missing_indices,
        "duplicate_source_indices": duplicate_source_indices,
        "true_error_indices": true_error_indices,
        "chosen_sources": dict(chosen_sources),
        "source_summaries": source_summaries,
    }

# -----------------------------
# First clean pass
# -----------------------------
first_pass = build_clean_from_candidates()

print("First clean pass")
print("Covered:", len(first_pass["clean_items"]), "/", len(chunks))
print("Missing:", len(first_pass["missing_indices"]))
print("True errors:", len(first_pass["true_error_indices"]))
print("True error chunk numbers:", summarize_ranges([i + 1 for i in first_pass["true_error_indices"]]))
print("Duplicate source indices ignored by merge:", len(first_pass["duplicate_source_indices"]))
print("Chosen sources:", first_pass["chosen_sources"])

# -----------------------------
# Safe retry schema
# -----------------------------
SAFE_KG_SCHEMA = copy.deepcopy(KG_SCHEMA)

SAFE_KG_SCHEMA["properties"]["entities"]["maxItems"] = 20
SAFE_KG_SCHEMA["properties"]["entities"]["items"]["maxLength"] = 120

SAFE_KG_SCHEMA["properties"]["relations"]["maxItems"] = 24
SAFE_KG_SCHEMA["properties"]["relations"]["items"]["properties"]["head"]["maxLength"] = 120
SAFE_KG_SCHEMA["properties"]["relations"]["items"]["properties"]["tail"]["maxLength"] = 120
SAFE_KG_SCHEMA["properties"]["relations"]["items"]["properties"]["relation"]["maxLength"] = 260

SAFE_KG_SCHEMA["properties"]["facts"]["maxItems"] = 20
SAFE_KG_SCHEMA["properties"]["facts"]["items"]["properties"]["entity"]["maxLength"] = 120
SAFE_KG_SCHEMA["properties"]["facts"]["items"]["properties"]["info"]["maxLength"] = 260

def build_safe_prompt(chunk):
    # Short prompt to reduce looping.
    return f"""Extract a compact knowledge graph from the Wikipedia chunk.

Return JSON only.
Do not explain.
Do not write analysis.
Do not use first person.
Do not repeat text.
Use only facts explicitly stated in the chunk.

Limits:
- entities: at most 20 short strings
- relations: at most 24 objects
- facts: at most 20 objects
- every relation head/tail must exactly match an entity
- every fact entity must exactly match an entity

JSON keys:
entities, relations, facts

Chunk id:
{chunk["Chunk_id"]}

Title:
{chunk["Title"]}

Paragraph ids:
{json.dumps(chunk["Paragraph_id"], ensure_ascii=False)}

Text:
{chunk["Text"]}
"""

def call_llm_safe_once(chunk, max_tokens):
    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {
                "role": "system",
                "content": "Return only compact valid JSON. No explanations."
            },
            {
                "role": "user",
                "content": build_safe_prompt(chunk)
            },
        ],
        max_tokens=max_tokens,
        temperature=0.0,
        top_p=1.0,
        presence_penalty=0.0,
        seed=123,
        extra_body={
            "top_k": 20,
            "min_p": 0.0,
            "repetition_penalty": 1.18,
            "chat_template_kwargs": {
                "enable_thinking": False
            },
            "structured_outputs": {
                "json": SAFE_KG_SCHEMA
            }
        },
    )

    choice = response.choices[0]

    return {
        "content": choice.message.content,
        "finish_reason": choice.finish_reason,
        "usage": usage_to_dict(response.usage),
        "max_tokens": max_tokens,
    }

def validate_safe_model_output(model_output):
    # Basic local validation.
    if not isinstance(model_output, dict):
        raise ValueError("Model output is not a dict.")

    for key in ["entities", "relations", "facts"]:
        if key not in model_output:
            raise ValueError(f"Missing key: {key}")

    entities = model_output["entities"]
    relations = model_output["relations"]
    facts = model_output["facts"]

    if not isinstance(entities, list):
        raise ValueError("entities is not a list.")

    if not isinstance(relations, list):
        raise ValueError("relations is not a list.")

    if not isinstance(facts, list):
        raise ValueError("facts is not a list.")

    entity_set = set(entities)

    for ent in entities:
        if not isinstance(ent, str) or not ent.strip():
            raise ValueError("Invalid entity string.")

        if len(ent) > 120:
            raise ValueError("Entity string too long.")

    for rel in relations:
        if not isinstance(rel, dict):
            raise ValueError("Relation is not a dict.")

        if rel.get("head") not in entity_set:
            raise ValueError("Relation head not in entities.")

        if rel.get("tail") not in entity_set:
            raise ValueError("Relation tail not in entities.")

        if not isinstance(rel.get("relation"), str) or not rel["relation"].strip():
            raise ValueError("Invalid relation text.")

        if len(rel["relation"]) > 260:
            raise ValueError("Relation text too long.")

    for fact in facts:
        if not isinstance(fact, dict):
            raise ValueError("Fact is not a dict.")

        if fact.get("entity") not in entity_set:
            raise ValueError("Fact entity not in entities.")

        if not isinstance(fact.get("info"), str) or not fact["info"].strip():
            raise ValueError("Invalid fact text.")

        if len(fact["info"]) > 260:
            raise ValueError("Fact text too long.")

def extract_one_safe(chunk, max_retries_per_budget=1):
    # Safer extraction for pathological chunks.
    start = time.perf_counter()
    attempts = []
    last_error = None
    raw = None
    last_finish_reason = None
    last_usage = None
    last_max_tokens = None

    for max_tokens in [2048, 3072]:
        for retry_id in range(max_retries_per_budget + 1):
            try:
                response_data = call_llm_safe_once(chunk, max_tokens=max_tokens)

                raw = response_data["content"]
                finish_reason = response_data["finish_reason"]
                usage = response_data["usage"]

                last_finish_reason = finish_reason
                last_usage = usage
                last_max_tokens = max_tokens

                attempts.append({
                    "max_tokens": max_tokens,
                    "retry_id": retry_id,
                    "finish_reason": finish_reason,
                    "usage": usage,
                })

                if finish_reason == "length":
                    last_error = f"finish_reason=length at max_tokens={max_tokens}"
                    break

                model_output = json.loads(raw)
                validate_safe_model_output(model_output)

                return {
                    "chunk_id": chunk["Chunk_id"],
                    "title": chunk["Title"],
                    "paragraph_id": chunk["Paragraph_id"],
                    "token_count": chunk.get("Token_count"),
                    "entities": model_output["entities"],
                    "relations": model_output["relations"],
                    "facts": model_output["facts"],
                    "error": None,
                    "latency_sec": round(time.perf_counter() - start, 3),
                    "finish_reason": finish_reason,
                    "max_tokens_used": max_tokens,
                    "usage": usage,
                    "attempts": attempts,
                    "safe_retry": True,
                }

            except Exception as e:
                last_error = repr(e)
                time.sleep(1.0 * (retry_id + 1))

    failed_item = {
        "chunk_id": chunk["Chunk_id"],
        "title": chunk["Title"],
        "paragraph_id": chunk["Paragraph_id"],
        "token_count": chunk.get("Token_count"),
        "entities": None,
        "relations": None,
        "facts": None,
        "error": last_error,
        "latency_sec": round(time.perf_counter() - start, 3),
        "finish_reason": last_finish_reason,
        "max_tokens_used": last_max_tokens,
        "usage": last_usage,
        "attempts": attempts,
        "safe_retry": True,
    }

    failed_item["raw_response"] = raw
    return failed_item

def load_existing_safe_retry_items():
    # Prefer final safe retry, then checkpoint.
    for path in [SAFE_REPAIR_OUTPUT_PATH, SAFE_REPAIR_PARTIAL_PATH]:
        if path.exists():
            try:
                return load_json_list(path)
            except Exception:
                pass

    return []

def save_safe_retry_partial(results_by_index):
    # Save completed safe retry items.
    atomic_json_dump(
        [results_by_index[idx] for idx in sorted(results_by_index)],
        SAFE_REPAIR_PARTIAL_PATH,
    )

# -----------------------------
# Retry true errors only
# -----------------------------
safe_retry_indices = first_pass["true_error_indices"]

if RUN_SAFE_RETRY and safe_retry_indices:
    print("\nSafe retry")
    print("Items to retry:", len(safe_retry_indices))
    print("Chunk numbers:", summarize_ranges([i + 1 for i in safe_retry_indices]))

    safe_results_by_index = {}

    for item in load_existing_safe_retry_items():
        idx = input_index_from_item(item)

        if idx in safe_retry_indices and item.get("error") is None:
            safe_results_by_index[idx] = item

    pending_safe_jobs = [
        (idx, chunks[idx])
        for idx in safe_retry_indices
        if idx not in safe_results_by_index
    ]

    print("Already successful safe retry items:", len(safe_results_by_index))
    print("Pending safe retry items:", len(pending_safe_jobs))

    safe_start = time.perf_counter()
    completed_new = 0

    if pending_safe_jobs:
        with ThreadPoolExecutor(max_workers=SAFE_MAX_WORKERS) as executor:
            futures = {
                executor.submit(extract_one_safe, chunk): (idx, chunk)
                for idx, chunk in pending_safe_jobs
            }

            for future in tqdm(
                as_completed(futures),
                total=len(futures),
                desc="Safe retry remaining errors"
            ):
                idx, chunk = futures[future]

                try:
                    item = future.result()
                except Exception as e:
                    item = {
                        "chunk_id": chunk["Chunk_id"],
                        "title": chunk["Title"],
                        "paragraph_id": chunk["Paragraph_id"],
                        "token_count": chunk.get("Token_count"),
                        "entities": None,
                        "relations": None,
                        "facts": None,
                        "error": repr(e),
                        "latency_sec": None,
                        "finish_reason": None,
                        "max_tokens_used": None,
                        "usage": None,
                        "attempts": [],
                        "safe_retry": True,
                    }

                item["input_index"] = idx
                item["repair_run"] = True
                item["repair_kind"] = "safe_retry"

                safe_results_by_index[idx] = item
                completed_new += 1

                if completed_new % SAFE_SAVE_EVERY == 0:
                    save_safe_retry_partial(safe_results_by_index)

        save_safe_retry_partial(safe_results_by_index)

    safe_retry_results = [
        safe_results_by_index[idx]
        for idx in sorted(safe_results_by_index)
    ]

    safe_retry_errors = [
        x["input_index"]
        for x in safe_retry_results
        if x.get("error") is not None
    ]

    safe_meta = {
        "dataset": "hotpotqa",
        "model": MODEL_NAME,
        "num_safe_retry_items": len(safe_retry_results),
        "num_safe_retry_errors": len(safe_retry_errors),
        "safe_retry_error_chunk_numbers": summarize_ranges([i + 1 for i in safe_retry_errors]),
        "safe_schema": "compact_with_string_maxLength",
        "temperature": 0.0,
        "top_p": 1.0,
        "repetition_penalty": 1.18,
        "max_workers": SAFE_MAX_WORKERS,
        "total_time_sec": round(time.perf_counter() - safe_start, 3),
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
    }

    atomic_json_dump(safe_retry_results, SAFE_REPAIR_OUTPUT_PATH)
    atomic_json_dump(safe_meta, SAFE_REPAIR_META_PATH)
    save_safe_retry_partial(safe_results_by_index)

    print("Saved safe retry JSON:", SAFE_REPAIR_OUTPUT_PATH)
    print("Safe retry remaining errors:", len(safe_retry_errors))
    print("Safe retry remaining error chunk numbers:", summarize_ranges([i + 1 for i in safe_retry_errors]))

# -----------------------------
# Final clean pass after safe retry
# -----------------------------
final_pass = build_clean_from_candidates()

final_clean_items = final_pass["clean_items"]
final_missing_indices = final_pass["missing_indices"]
final_true_error_indices = final_pass["true_error_indices"]

final_audit = {
    "dataset": "hotpotqa",
    "input_path": str(GDRIVE_INPUT_PATH),
    "kg_dir": str(KG_DIR),
    "clean_output_path": str(CLEAN_OUTPUT_PATH),
    "total_chunks": len(chunks),
    "num_clean_items": len(final_clean_items),
    "num_missing_indices": len(final_missing_indices),
    "num_true_error_indices": len(final_true_error_indices),
    "num_duplicate_source_indices_ignored": len(final_pass["duplicate_source_indices"]),
    "missing_input_index_ranges_0_based": summarize_ranges(final_missing_indices),
    "missing_chunk_number_ranges_1_based": summarize_ranges([i + 1 for i in final_missing_indices]),
    "true_error_input_index_ranges_0_based": summarize_ranges(final_true_error_indices),
    "true_error_chunk_number_ranges_1_based": summarize_ranges([i + 1 for i in final_true_error_indices]),
    "chosen_sources": final_pass["chosen_sources"],
    "source_summaries": final_pass["source_summaries"],
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
}

atomic_json_dump(final_clean_items, CLEAN_OUTPUT_PATH)
atomic_json_dump(final_audit, CLEAN_AUDIT_PATH)

print("\nFinal clean output")
print("Saved clean JSON:", CLEAN_OUTPUT_PATH)
print("Saved clean audit:", CLEAN_AUDIT_PATH)
print("Clean items:", len(final_clean_items), "/", len(chunks))
print("Missing:", len(final_missing_indices))
print("True errors:", len(final_true_error_indices))
print("True error chunk numbers:", summarize_ranges([i + 1 for i in final_true_error_indices]))
print("Duplicate source indices ignored:", len(final_pass["duplicate_source_indices"]))
print("Chosen sources:", final_pass["chosen_sources"])

if final_missing_indices:
    raise RuntimeError("Some chunks are still missing after clean merge.")

if final_true_error_indices:
    print("Warning: some chunks still failed after safe retry. Use the clean audit file to inspect them.")
else:
    print("All chunks are covered with one clean successful record per input_index.")

First clean pass
Covered: 35029 / 35029
Missing: 0
True errors: 16
True error chunk numbers: ['3182', '5449', '6516', '6947-6948', '7536', '10286', '12953', '13113', '16563', '19278', '19603', '20332', '21074', '25572', '29833']
Duplicate source indices ignored by merge: 19
Chosen sources: {'original_batch': 34981, 'repair_v1': 48}

Safe retry
Items to retry: 16
Chunk numbers: ['3182', '5449', '6516', '6947-6948', '7536', '10286', '12953', '13113', '16563', '19278', '19603', '20332', '21074', '25572', '29833']
Already successful safe retry items: 0
Pending safe retry items: 16


Safe retry remaining errors:   0%|          | 0/16 [00:00<?, ?it/s]

Saved safe retry JSON: /content/drive/MyDrive/final_project/idea_1/kg/hotpotqa_kg_extractions_repair_safe_retry.json
Safe retry remaining errors: 5
Safe retry remaining error chunk numbers: ['5449', '6947-6948', '7536', '25572']

Final clean output
Saved clean JSON: /content/drive/MyDrive/final_project/idea_1/kg/hotpotqa_kg_extractions_all_00000001_to_00035029.clean.json
Saved clean audit: /content/drive/MyDrive/final_project/idea_1/kg/hotpotqa_kg_extractions_all_00000001_to_00035029.clean_audit.json
Clean items: 35029 / 35029
Missing: 0
True errors: 5
True error chunk numbers: ['5449', '6947-6948', '7536', '25572']
Duplicate source indices ignored: 19
Chosen sources: {'original_batch': 34981, 'repair_v1': 32, 'repair_safe_retry': 16}
